# Car Price Predictions 2025

In [3]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go

In [5]:
cars = pd.read_csv("../data/car_price_prediction_.csv")

In [6]:
display(cars.head())

print ("=" * 40)

display(cars.info())

print ("=" * 40)

display(cars.describe())

print ("=" * 40)

print(f"Dataset contains {cars.shape[0]} rows and {cars.shape[1]} columns.")

display(cars.isnull().sum())
print ("=" * 40)

,Car ID,Brand,Year,Engine Size,Fuel Type,Transmission,Mileage,Condition,Price,Model
0,1,Tesla,2016,2.3,Petrol,Manual,114832,New,26613.92,Model X
1,2,BMW,2018,4.4,Electric,Manual,143190,Used,14679.61,5 Series
2,3,Audi,2013,4.5,Electric,Manual,181601,New,44402.61,A4
3,4,Tesla,2011,4.1,Diesel,Automatic,68682,New,86374.33,Model Y
4,5,Ford,2009,2.6,Diesel,Manual,223009,Like New,73577.10,Mustang


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Car ID        2500 non-null   int64  
 1   Brand         2500 non-null   object 
 2   Year          2500 non-null   int64  
 3   Engine Size   2500 non-null   float64
 4   Fuel Type     2500 non-null   object 
 5   Transmission  2500 non-null   object 
 6   Mileage       2500 non-null   int64  
 7   Condition     2500 non-null   object 
 8   Price         2500 non-null   float64
 9   Model         2500 non-null   object 
dtypes: float64(2), int64(3), object(5)
memory usage: 195.4+ KB


None

,Car ID,Year,Engine Size,Mileage,Price
count,2500.00000,2500.0000,2500.000000,2500.000000,2500.000000
mean,1250.50000,2011.6268,3.465240,149749.844800,52638.022532
std,721.83216,6.9917,1.432053,87919.952034,27295.833455
min,1.00000,2000.0000,1.000000,15.000000,5011.270000
25%,625.75000,2005.0000,2.200000,71831.500000,28908.485000
50%,1250.50000,2012.0000,3.400000,149085.000000,53485.240000
75%,1875.25000,2018.0000,4.700000,225990.500000,75838.532500
max,2500.00000,2023.0000,6.000000,299967.000000,99982.590000


Dataset contains 2500 rows and 10 columns.


Car ID          0
Brand           0
Year            0
Engine Size     0
Fuel Type       0
Transmission    0
Mileage         0
Condition       0
Price           0
Model           0
dtype: int64

In [8]:
# distribution of car brands
fig = px.histogram(cars, x='Brand', title='Distribution of Car Brands', labels={'Brand': 'Car Brand', 'count': 'Number of Cars'}, template='plotly_dark')
fig.update_layout(bargap=0.2)
fig.show()

# avg price for brand
fig = px.bar(cars.groupby('Brand')['Price'].mean().reset_index(), x='Brand', y='Price', title='Average Price for Car Brands', labels={'Brand': 'Car Brand', 'Price': 'Average Price'}, template='plotly_dark')
fig.show()

### **Predictive analysis and comparation**

**Feature Engineering**

In [13]:
cars['Car_Age'] = 2025 - cars['Year']
cars["Mileage_per_Year"] = cars["Mileage"] / (cars["Car_Age"] + 1)

luxury_brands = ["Tesla", "BMW", "Audi", "Mercedes"]
cars["Is_Luxury"] = cars["Brand"].isin(luxury_brands).astype(int)

cars["Fuel_Group"] = cars["Fuel Type"].replace({"Petrol": "Traditional", "Diesel": "Traditional", "Hybrid": "Eco", "Electric": "Eco"})

**Encoding**

In [15]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer, make_column_transformer

In [19]:
# Split data before encoding
X = cars.drop(["Price", "Car ID", "Model"], axis=1)
y = cars["Price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=69)

In [20]:
# Column transformer for encoding and scaling
categorical_features = ['Brand', 'Fuel Type', 'Transmission', 'Fuel_Group']
numerical_features = ['Year', 'Mileage', 'Car_Age', 'Mileage_per_Year', 'Is_Luxury']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(), categorical_features)
    ]
)
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

### **Predictive analysis and comparation**

In [23]:
# we will be trying different models hereafter

# Lasso Regression
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Forest and Gradient Boosting Regressors
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# XGBoost Regressor
from xgboost import XGBRegressor

In [24]:
# Lasso Regression
lasso = Lasso(alpha=0.1)
lasso.fit(X_train, y_train)
y_pred_lasso = lasso.predict(X_test)

print("Lasso Regression Performance:")
print(f"MAE: {mean_absolute_error(y_test, y_pred_lasso)}")
print(f"MSE: {mean_squared_error(y_test, y_pred_lasso)}")
print(f"R²: {r2_score(y_test, y_pred_lasso)}")

Lasso Regression Performance:
MAE: 23857.055889789848
MSE: 748727538.6663694
R²: -0.01713570652801577


In [25]:
# Forest and Gradient Boosting Regressors
rf = RandomForestRegressor(n_estimators=100, random_state=69)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("\nRandom Forest Regressor Performance:")
print(f"MAE: {mean_absolute_error(y_test, y_pred_rf)}")
print(f"MSE: {mean_squared_error(y_test, y_pred_rf)}")
print(f"R²: {r2_score(y_test, y_pred_rf)}")

gb = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=69)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)
print("\nGradient Boosting Regressor Performance:")
print(f"MAE: {mean_absolute_error(y_test, y_pred_gb)}")
print(f"MSE: {mean_squared_error(y_test, y_pred_gb)}")
print(f"R²: {r2_score(y_test, y_pred_gb)}")



Random Forest Regressor Performance:
MAE: 24705.570646
MSE: 828088930.8557229
R²: -0.1249470284667622

Gradient Boosting Regressor Performance:
MAE: 24024.814828325285
MSE: 766741628.5671117
R²: -0.041607591309076764


In [26]:
# XGBoost Regressor
xgb = XGBRegressor(n_estimators=100, learning_rate=0.1, objective='reg:squarederror', random_state=69)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)
print("\nXGBoost Regressor Performance:")
print(f"MAE: {mean_absolute_error(y_test, y_pred_xgb)}")
print(f"MSE: {mean_squared_error(y_test, y_pred_xgb)}")
print(f"R²: {r2_score(y_test, y_pred_xgb)}")


XGBoost Regressor Performance:
MAE: 24572.1698203125
MSE: 834487080.8517034
R²: -0.13363882418757322


Comparisons

In [28]:
print("\nComparisons:")
models = ['Lasso Regression', 'Random Forest', 'Gradient Boosting', 'XGBoost']
mae_values = [mean_absolute_error(y_test, y_pred_lasso),
              mean_absolute_error(y_test, y_pred_rf),
              mean_absolute_error(y_test, y_pred_gb),
              mean_absolute_error(y_test, y_pred_xgb)]
mse_values = [mean_squared_error(y_test, y_pred_lasso),
              mean_squared_error(y_test, y_pred_rf),
              mean_squared_error(y_test, y_pred_gb),
              mean_squared_error(y_test, y_pred_xgb)]
r2_values = [r2_score(y_test, y_pred_lasso),
             r2_score(y_test, y_pred_rf),
             r2_score(y_test, y_pred_gb),
             r2_score(y_test, y_pred_xgb)]

comparison_df = pd.DataFrame({
    'Model': models,
    'MAE': mae_values,
    'MSE': mse_values,
    'R²': r2_values
})

display(comparison_df)

# normalize for better visualization
comparison_df['MAE'] = comparison_df['MAE'] / comparison_df['MAE'].max()
comparison_df['MSE'] = comparison_df['MSE'] / comparison_df['MSE'].max()
comparison_df['R²'] = comparison_df['R²'] / comparison_df['R²'].max()

# plot comparison
fig = go.Figure(data=[
    go.Bar(name='MAE', x=comparison_df['Model'], y=comparison_df['MAE']),
    go.Bar(name='MSE', x=comparison_df['Model'], y=comparison_df['MSE']),
    go.Bar(name='R²', x=comparison_df['Model'], y=comparison_df['R²'])
])

fig.update_layout(barmode='group', title='Model Performance Comparison', yaxis_title='Value', template='plotly_dark')
fig.show()


Comparisons:


,Model,MAE,MSE,R²
0,Lasso Regression,23857.055890,7.487275e+08,-0.017136
1,Random Forest,24705.570646,8.280889e+08,-0.124947
2,Gradient Boosting,24024.814828,7.667416e+08,-0.041608
3,XGBoost,24572.169820,8.344871e+08,-0.133639


In [29]:
# Model Serialization
import joblib
import os

# Create models directory if it doesn't exist
models_dir = '../models'
if not os.path.exists(models_dir):
    os.makedirs(models_dir)

# Save the best model (XGBoost)
model_path = os.path.join(models_dir, 'car_price_xgboost_model.joblib')
joblib.dump(xgb, model_path)
print(f"✓ XGBoost model saved to: {model_path}")

# Save the preprocessor
preprocessor_path = os.path.join(models_dir, 'car_price_preprocessor.joblib')
joblib.dump(preprocessor, preprocessor_path)
print(f"✓ Preprocessor saved to: {preprocessor_path}")

# Save feature names and structure
feature_info = {
    'categorical_features': categorical_features,
    'numerical_features': numerical_features,
    'all_features': X.columns.tolist()
}
feature_info_path = os.path.join(models_dir, 'car_price_feature_info.joblib')
joblib.dump(feature_info, feature_info_path)
print(f"✓ Feature information saved to: {feature_info_path}")

✓ XGBoost model saved to: ../models/car_price_xgboost_model.joblib
✓ Preprocessor saved to: ../models/car_price_preprocessor.joblib
✓ Feature information saved to: ../models/car_price_feature_info.joblib


In [30]:
# Prediction Function for New Cars
def predict_car_price(year, brand, fuel_type, transmission, mileage):
    """
    Predict car price using the trained XGBoost model.
    
    Parameters:
    -----------
    year : int
        Year of manufacture (e.g., 2020, 2023, 2024)
    brand : str
        Car brand (e.g., 'Toyota', 'BMW', 'Tesla', 'Honda')
    fuel_type : str
        Fuel type ('Petrol', 'Diesel', 'Hybrid', 'Electric')
    transmission : str
        Transmission type ('Manual', 'Automatic')
    mileage : float
        Current mileage in kilometers
    
    Returns:
    --------
    float : Predicted price in currency units
    """
    
    # Calculate derived features
    current_year = 2025
    car_age = current_year - year
    mileage_per_year = mileage / (car_age + 1)
    
    # Determine if luxury brand
    luxury_brands = ["Tesla", "BMW", "Audi", "Mercedes"]
    is_luxury = 1 if brand in luxury_brands else 0
    
    # Determine fuel group
    fuel_group_map = {
        "Petrol": "Traditional",
        "Diesel": "Traditional",
        "Hybrid": "Eco",
        "Electric": "Eco"
    }
    fuel_group = fuel_group_map.get(fuel_type, "Traditional")
    
    # Create input dataframe with the same structure as training data
    input_data = pd.DataFrame({
        'Brand': [brand],
        'Year': [year],
        'Mileage': [mileage],
        'Car_Age': [car_age],
        'Mileage_per_Year': [mileage_per_year],
        'Is_Luxury': [is_luxury],
        'Fuel Type': [fuel_type],
        'Transmission': [transmission],
        'Fuel_Group': [fuel_group]
    })
    
    # Apply the same preprocessing
    input_transformed = preprocessor.transform(input_data)
    
    # Make prediction
    predicted_price = xgb.predict(input_transformed)[0]
    
    return predicted_price

# Test the prediction function with sample cars
print("\n" + "=" * 80)
print("CAR PRICE PREDICTION FUNCTION - SAMPLE PREDICTIONS")
print("=" * 80)

test_cars = [
    {
        "name": "2023 Toyota Camry",
        "year": 2023,
        "brand": "Toyota",
        "fuel_type": "Petrol",
        "transmission": "Automatic",
        "mileage": 15000
    },
    {
        "name": "2020 BMW X5",
        "year": 2020,
        "brand": "BMW",
        "fuel_type": "Diesel",
        "transmission": "Automatic",
        "mileage": 45000
    },
    {
        "name": "2024 Tesla Model 3",
        "year": 2024,
        "brand": "Tesla",
        "fuel_type": "Electric",
        "transmission": "Automatic",
        "mileage": 5000
    },
    {
        "name": "2019 Honda Civic",
        "year": 2019,
        "brand": "Honda",
        "fuel_type": "Petrol",
        "transmission": "Manual",
        "mileage": 60000
    },
    {
        "name": "2022 Audi A6 Hybrid",
        "year": 2022,
        "brand": "Audi",
        "fuel_type": "Hybrid",
        "transmission": "Automatic",
        "mileage": 25000
    }
]

for i, car in enumerate(test_cars, 1):
    car_name = car.pop("name")
    predicted_price = predict_car_price(**car)
    print(f"\n{i}. {car_name}")
    print(f"   Year: {car['year']} | Brand: {car['brand']} | Fuel: {car['fuel_type']} | Transmission: {car['transmission']}")
    print(f"   Mileage: {car['mileage']:,} km")
    print(f"   → Predicted Price: ₹{predicted_price:,.2f}")

print("\n" + "=" * 80)


CAR PRICE PREDICTION FUNCTION - SAMPLE PREDICTIONS

1. 2023 Toyota Camry
   Year: 2023 | Brand: Toyota | Fuel: Petrol | Transmission: Automatic
   Mileage: 15,000 km
   → Predicted Price: ₹44,110.74

2. 2020 BMW X5
   Year: 2020 | Brand: BMW | Fuel: Diesel | Transmission: Automatic
   Mileage: 45,000 km
   → Predicted Price: ₹62,744.34

3. 2024 Tesla Model 3
   Year: 2024 | Brand: Tesla | Fuel: Electric | Transmission: Automatic
   Mileage: 5,000 km
   → Predicted Price: ₹56,774.66

4. 2019 Honda Civic
   Year: 2019 | Brand: Honda | Fuel: Petrol | Transmission: Manual
   Mileage: 60,000 km
   → Predicted Price: ₹54,815.82

5. 2022 Audi A6 Hybrid
   Year: 2022 | Brand: Audi | Fuel: Hybrid | Transmission: Automatic
   Mileage: 25,000 km
   → Predicted Price: ₹46,671.50



In [31]:
# Input Validation and Error Handling
class CarPricePredictor:
    """
    Complete car price prediction system with validation and error handling.
    """
    
    def __init__(self, model_path, preprocessor_path, feature_info_path):
        """
        Initialize the predictor with saved model and preprocessor.
        
        Parameters:
        -----------
        model_path : str
            Path to saved XGBoost model
        preprocessor_path : str
            Path to saved preprocessor
        feature_info_path : str
            Path to saved feature information
        """
        self.model = joblib.load(model_path)
        self.preprocessor = joblib.load(preprocessor_path)
        self.feature_info = joblib.load(feature_info_path)
        
        self.valid_brands = ["Toyota", "Honda", "BMW", "Audi", "Mercedes", "Tesla", "Maruti", "Hyundai", "Ford", "Mahindra"]
        self.valid_fuel_types = ["Petrol", "Diesel", "Hybrid", "Electric"]
        self.valid_transmissions = ["Manual", "Automatic"]
        
    def validate_input(self, year, brand, fuel_type, transmission, mileage):
        """Validate input parameters."""
        errors = []
        
        current_year = 2025
        if year < 1990 or year > current_year:
            errors.append(f"Year must be between 1990 and {current_year}")
        
        if brand not in self.valid_brands:
            errors.append(f"Brand must be one of: {', '.join(self.valid_brands)}")
        
        if fuel_type not in self.valid_fuel_types:
            errors.append(f"Fuel type must be one of: {', '.join(self.valid_fuel_types)}")
        
        if transmission not in self.valid_transmissions:
            errors.append(f"Transmission must be one of: {', '.join(self.valid_transmissions)}")
        
        if mileage < 0:
            errors.append("Mileage cannot be negative")
        
        if errors:
            return False, errors
        return True, []
    
    def predict(self, year, brand, fuel_type, transmission, mileage):
        """
        Predict car price with validation.
        
        Returns:
        --------
        dict : Prediction result with price and confidence metrics
        """
        # Validate input
        is_valid, errors = self.validate_input(year, brand, fuel_type, transmission, mileage)
        if not is_valid:
            return {
                "status": "error",
                "errors": errors,
                "price": None
            }
        
        # Calculate derived features
        current_year = 2025
        car_age = current_year - year
        mileage_per_year = mileage / (car_age + 1)
        
        # Determine if luxury brand
        luxury_brands = ["Tesla", "BMW", "Audi", "Mercedes"]
        is_luxury = 1 if brand in luxury_brands else 0
        
        # Determine fuel group
        fuel_group_map = {
            "Petrol": "Traditional",
            "Diesel": "Traditional",
            "Hybrid": "Eco",
            "Electric": "Eco"
        }
        fuel_group = fuel_group_map.get(fuel_type, "Traditional")
        
        # Create input dataframe
        input_data = pd.DataFrame({
            'Brand': [brand],
            'Year': [year],
            'Mileage': [mileage],
            'Car_Age': [car_age],
            'Mileage_per_Year': [mileage_per_year],
            'Is_Luxury': [is_luxury],
            'Fuel Type': [fuel_type],
            'Transmission': [transmission],
            'Fuel_Group': [fuel_group]
        })
        
        # Apply preprocessing
        input_transformed = self.preprocessor.transform(input_data)
        
        # Make prediction
        predicted_price = self.model.predict(input_transformed)[0]
        
        return {
            "status": "success",
            "year": year,
            "brand": brand,
            "fuel_type": fuel_type,
            "transmission": transmission,
            "mileage": mileage,
            "predicted_price": round(predicted_price, 2),
            "car_age": car_age,
            "is_luxury": is_luxury
        }

# Initialize the predictor
print("\nInitializing Car Price Predictor...")
predictor = CarPricePredictor(
    model_path='../models/car_price_xgboost_model.joblib',
    preprocessor_path='../models/car_price_preprocessor.joblib',
    feature_info_path='../models/car_price_feature_info.joblib'
)
print("✓ Predictor initialized successfully!")

# Test with sample predictions
print("\n" + "=" * 80)
print("CAR PRICE PREDICTOR - ROBUST SYSTEM TEST")
print("=" * 80)

test_cases = [
    {"year": 2023, "brand": "Toyota", "fuel_type": "Petrol", "transmission": "Automatic", "mileage": 15000},
    {"year": 2020, "brand": "BMW", "fuel_type": "Diesel", "transmission": "Automatic", "mileage": 45000},
    {"year": 2024, "brand": "Tesla", "fuel_type": "Electric", "transmission": "Automatic", "mileage": 5000},
    {"year": 2019, "brand": "Honda", "fuel_type": "Petrol", "transmission": "Manual", "mileage": 60000},
]

for i, case in enumerate(test_cases, 1):
    result = predictor.predict(**case)
    
    if result["status"] == "success":
        print(f"\n{i}. {result['year']} {result['brand']}")
        print(f"   Fuel: {result['fuel_type']} | Transmission: {result['transmission']}")
        print(f"   Mileage: {result['mileage']:,} km | Age: {result['car_age']} years")
        print(f"   Luxury: {'Yes' if result['is_luxury'] else 'No'}")
        print(f"   → Predicted Price: ₹{result['predicted_price']:,.2f}")
    else:
        print(f"\n{i}. Prediction Error:")
        for error in result["errors"]:
            print(f"   - {error}")

print("\n" + "=" * 80)


Initializing Car Price Predictor...
✓ Predictor initialized successfully!

CAR PRICE PREDICTOR - ROBUST SYSTEM TEST

1. 2023 Toyota
   Fuel: Petrol | Transmission: Automatic
   Mileage: 15,000 km | Age: 2 years
   Luxury: No
   → Predicted Price: ₹44,110.74

2. 2020 BMW
   Fuel: Diesel | Transmission: Automatic
   Mileage: 45,000 km | Age: 5 years
   Luxury: Yes
   → Predicted Price: ₹62,744.34

3. 2024 Tesla
   Fuel: Electric | Transmission: Automatic
   Mileage: 5,000 km | Age: 1 years
   Luxury: Yes
   → Predicted Price: ₹56,774.66

4. 2019 Honda
   Fuel: Petrol | Transmission: Manual
   Mileage: 60,000 km | Age: 6 years
   Luxury: No
   → Predicted Price: ₹54,815.82

✓ Predictor initialized successfully!

CAR PRICE PREDICTOR - ROBUST SYSTEM TEST

1. 2023 Toyota
   Fuel: Petrol | Transmission: Automatic
   Mileage: 15,000 km | Age: 2 years
   Luxury: No
   → Predicted Price: ₹44,110.74

2. 2020 BMW
   Fuel: Diesel | Transmission: Automatic
   Mileage: 45,000 km | Age: 5 years
   Lu